In [112]:
#!pip install --upgrade gensim
#!pip install -U spacy
%load_ext autoreload
%autoreload 2
from gensim.matutils import dense2vec, Dense2Corpus, corpus2dense
from gensim.models import KeyedVectors
import numpy as np
import pandas as pd
from model import *
from datasets import load_data
from utils_model import *
from prepro_txt import *
#from utilsembedding import *
import eli5
import re
#from utilsembedding_eff import *
#from explainability_global import get_subspace_of_doc
from collections import Counter
import spacy
from typing import List
#!pip install lime
from lime import lime_text
from lime.lime_text import LimeTextExplainer
from sklearn.pipeline import make_pipeline
#from utilsembedding import tokenizer, pretrained_models_gensim, load_model_gensim, get_doc_embedding, get_doc_subspace, get_docs_subspaces#, preprocessing
NLP = spacy.blank('en')
STOPWORDS = stopwords.words('english')
NLP.max_length= 4000000
EMBEDDING_DIM = 300
## ISSUE ONLY WITH HYPERPARTISAN 
dataname = 'reuters-8' #dataname='hyperpartisan'
class_names={'reuters-8': ['acq','crude', 'earn', 'grain', 'interest', 'money-fx', 'ship', 'trade'],
             'hyperpartisan': ['true', 'false']}
d_num={'reuters-8': 20, 'hyperpartisan': 30}
lr_w ={'reuters-8':0.01, 'hyperpartisan': 0.1}  # reuters8 (10d) 0.01
lr_r = 0.00001  # imdb20d: 0.1, 0.000001
num_of_epochs = 100
distance_type, act_fun, sigma, balanced='pseudo-chordal', 'sigmoid', 100, False
subspace_dim=d_num[dataname]

def load_model_gensim(checkpoint):
    return KeyedVectors.load('../model/%s.d2v' % checkpoint)
    
MODEL = load_model_gensim('glove.42B.300d') # 'word2vec-google-news-300')
w = MODEL.most_similar('twitter', topn=10)[np.random.randint(10)][0]
print(w)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
blogs


In [81]:
class preprocessText:
    def __init__(self, subspace_dim, emb_size, model_name):
        self.subspace_dim=subspace_dim
        self.embsize=emb_size
        self.weighted=True
        self.sing_values=False
        self.model_name=model_name
        
    def transform(self, corpus_txt):         
        weighted, dim=True, subspace_dim
        m_name=self.model_name
        MODEL, MIN_LENGTH = load_model_gensim(m_name), 3
        if isinstance(corpus_txt, list):            
            corpus_txt=corpus_txt[0]
            print('\n corpus_txt[0], and its type and len: ', corpus_txt, type(corpus_txt), len(corpus_txt))
        NLP = spacy.load("en_core_web_sm", exclude=["tok2vec", "parser", "ner", "attrbute_ruler"]) 
        corpus_tokens=[[word.lower_ for word in NLP(corpus_txt) if (word.lower_ not in STOPWORDS) and (len(word) >= MIN_LENGTH)]]
        c_word = Counter(corpus_tokens[0])        
        words = [word for word in c_word.keys() if word in MODEL.index_to_key]
        assert len(words) >= MIN_LENGTH, f"the length of the text {len(words)} is lower than the dimensionality of the subspace {subspace_dim}"
        words_emb = np.array([MODEL.get_vector(word) for word in words]).T
        freq_words = np.array([c_word[word] for word in words])
        F = np.diag(np.sqrt(freq_words))
        U, S, Vh = np.linalg.svd(words_emb @ F, full_matrices=False, compute_uv=True, hermitian=False)
        word_impact_no_rot = F @ Vh[:subspace_dim, :].T @ np.diag(1 / S[:subspace_dim])
        xval=U[:, :subspace_dim]
        #print(xval.shape)
        #xval = preprocessing(corpus_txt, dim,  m_name)#, self.sing_values)
        return xval

In [50]:
 #'newsgroups20'
#df_tr = pd.read_csv('../data/%s-train-all-terms.csv' % dataname)
Xtrain, Ytrain, Xval, Yval = load_data(dataname)
#xprotos, yprotos, lamda =load_results('../model/%s/%s_model_d20_ps' % (dataname, dataname))
fname='../model/%s/glove/%s_model_d%i_ps' % (dataname, dataname, d_num[dataname])
with np.load(fname + '.npz', allow_pickle=True) as f:
    xprotos, yprotos, lamda = f['xprotos'], f['yprotos'], f['lamda']        

if Xtrain.shape[-1] != subspace_dim:
    Xtrain = Xtrain[:, :, :subspace_dim]
    Xval = Xval[:, :, :subspace_dim]
modelLVQ = Model(
        dim_of_data=Xtrain.shape[-2],        # the dimensionality of data
        dim_of_subspace=Xtrain.shape[-1],    # number of data in a set
        num_of_classes=len(np.unique(Ytrain)),  # number of classes
        distance=distance_type,     # (pseudo) chordal or geodesic
        balanced=balanced,
        localized=False,
        nprotos=1,          # for now it is only 1: check if you need to modify it for more
        actfun=act_fun,# the function inside cost function: identity or sigmoid
        sigma=sigma,            # parameter for sigmoid function
        xprotos=xprotos,
        yprotos=yprotos,
        lamda=lamda,)
modelLVQ.initialize_parameters(classes=Ytrain)
modelLVQ.fit(Xtrain, Ytrain,lr_w=lr_w[dataname], lr_r=lr_r,)
#c = make_pipeline(model)
pred_train = modelLVQ.predict(Xtrain) #c.predict(token_tr)
prob_train= modelLVQ.predict_proba(Xtrain)#c.predict_proba(token_tr)
pred_val = modelLVQ.predict(Xval) #c.predict(token_val)
prob_val= modelLVQ.predict_proba(Xval)#c.predict_proba(token_val)
print('Accuracy: train & test: ', np.sum(pred_train==Ytrain)/len(Ytrain)*100, np.sum(pred_val==Yval)/len(Yval)*100)
print('Accuracy: train & test: (from model metrics)', modelLVQ.metrics(Ytrain, pred_train)[0], modelLVQ.metrics(Yval, pred_val)[0])
print(prob_val[0], pred_val[0])


weights [1. 1. 1. 1. 1. 1. 1. 1.] 

Accuracy: train & test:  97.22880583409298 95.56875285518501
Accuracy: train & test: (from model metrics) 97.22880583409298 95.56875285518501
[0.12246722 0.12418416 0.11763577 0.1222695  0.11755338 0.12417466
 0.12381243 0.14790289] 7.0


In [82]:
NLP.max_length= 4000000
EMBEDDING_DIM = int(300)
MIN_LENGTH = 3 # minimum number of character for a token
pre=preprocessText(20, EMBEDDING_DIM, 'glove.42B.300d') #(corpus_text: List[str], subspace_dim, model_name='word2vec-google-news-300')
#pre=preprocessing(20, EMBEDDING_DIM, MODEL)
df_te = pd.read_csv('../data/%s/%s-test.csv' % (dataname, dataname))
#corpus_tr, ytrain = df_tr['text'].tolist(), df_tr['label'].tolist()
#cat = df_tr['category'].tolist()X
idx=0
sample_text=df_te['text'].iloc[idx]
#corpus_val, yval = df_te['text'].tolist(), df_te['label'].tolist()
c = make_pipeline(pre, modelLVQ)

In [83]:
corpus_text= sample_text#df_te['text'].iloc[0]
corpus_tokens, MIN_LENGTH, subspace_dim = [], 3, 20
print(type(corpus_text), len(corpus_text))
# Exclude components not required when loading the spaCy model.
NLP = spacy.load("en_core_web_sm", exclude=["tok2vec", "parser", "ner", "attrbute_ruler"]) 
print(type(corpus_text))
# Extract lemmas as required. a = [[word.lemma_ for word in nlp(doc) if word.is_punct == False and word.is_stop == False] for doc in doc_list]
#for doc in NLP.pipe(corpus_text):
             #print('from tokenizer::utilsembedding type(doc) and type(corpus): ',type(doc), type(corpus[0]), type(corpus))
    #corpus_tokens.append([t.lower_ for t in doc if (t.is_alpha and (t.lower_ not in STOPWORDS) and (len(t) >= MIN_LENGTH))])
corpus_tokens=[[word.lower_ for word in NLP(corpus_text) if (word.lower_ not in STOPWORDS) and (len(word) >= MIN_LENGTH)]]
#corpus_tokens = tokenizer(corpus_text)
c_word=Counter(corpus_tokens[0])
words = [word for word in c_word.keys() if word in MODEL.index_to_key]
assert len(words) >= MIN_LENGTH, f"the length of the text {len(words)} is lower than the dimensionality of the subspace {subspace_dim}"
words_emb = np.array([MODEL.get_vector(word) for word in words]).T
freq_words = np.array([c_word[word] for word in words])
F = np.diag(np.sqrt(freq_words))
U, S, Vh = np.linalg.svd(words_emb @ F, full_matrices=False, compute_uv=True, hermitian=False)
word_impact_no_rot = F @ Vh[:subspace_dim, :].T @ np.diag(1 / S[:subspace_dim])
xval=U[:, :subspace_dim]
modelLVQ.fit(Xtrain, Ytrain,lr_w=lr_w[dataname], lr_r=lr_r,)
print(modelLVQ.predict(xval), xval.shape, type(xval))
#print(c.predict(sample_text))

<class 'str'> 3098
<class 'str'>
7.0 (300, 20) <class 'numpy.ndarray'>


In [84]:
print('\n', type(sample_text), c)
print(c.predict(sample_text), len(sample_text))


 <class 'str'> Pipeline(steps=[('preprocesstext',
                 <__main__.preprocessText object at 0x7df85530d810>),
                ('model', <model.Model object at 0x7df5b3b83a90>)])
7.0 3098


In [85]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model._base import LinearModel,RegressorMixin
class LemnaRegressor(LinearModel, RegressorMixin):#class SurrogateRegressor(DecisionTreeRegressor):
    def fit(self, X, y, sample_weight=None):
      # Make sure X's row represents the ordered set of words (features)
        n_samples=50
        print(X.shape)
        X=X.flatten()
        n_feat = X.shape
        print(X.shape)
        K = 2 # number of components in mixture
        eps = 10 ** -4
        # Expectation
        # start with random assignment to mixture components
        Z0 = np.random.randint(2, size=(n_samples, 1))
        Z_ = np.concatenate([Z0,1-Z0], axis = 1)
        Z = np.concatenate([1-Z0,Z0], axis = 1) # just make it different than Z
        itr = 0
        while np.any(Z != Z_):
            itr += 1
            print(itr)
            Z = Z_.copy()       
            # Minimization
            # split into two clusters
            X0 = X[np.where(Z[:,0]==1)]
            y0 = y[np.where(Z[:,0]==1)]
            w0 = sample_weight[np.where(Z[:,0]==1)]
            X1 = X[np.where(Z[:,1]==1)]
            y1 = y[np.where(Z[:,1]==1)]
            w1 = sample_weight[np.where(Z[:,1]==1)]            
            # compute a LR model with fused LASSO for each cluster
            beta0 = cp.Variable(n_feat)
            forwardDiff0 = beta0[1:] - beta0[0:-1]
            objective0 = 0.5 * cp.Minimize( cp.sum_squares( cp.multiply(w0, (X0 @ beta0 - y0) ) ) )
            constraints0 = [cp.norm(forwardDiff0,2) <= eps]
            prob0 = cp.Problem(objective0, constraints0)
            result0 = prob0.solve()
            beta1 = cp.Variable(n_feat)
            forwardDiff1 = beta1[1:] - beta1[0:-1]
            objective1 = 0.5 * cp.Minimize(cp.sum_squares( cp.multiply(w1, (X1 @ beta1 - y1) ) ) )
            constraints1 = [cp.norm(forwardDiff1,2) <= eps]
            prob1 = cp.Problem(objective1, constraints1)
            result1 = prob1.solve()
            
            # Expectation
            # reassign Z
            Z_ = np.reshape(np.abs(X @ beta0.value - y) < np.abs(X @ beta1.value - y), (n_samples, 1))
            Z_ = np.concatenate([Z_,1-Z_], axis = 1)
        
        print ("after ", itr, "iterations:")
        # find the original sample we're explaining (all ones)
        idx = np.where(X.all(1))[0][0]
        if Z[idx][0] == 1: # it belongs to cluster 0
            self.coef_ = beta0.value
        else: # it belongs to cluster 1
            self.coef_ = beta1.value
            self.intercept_ = 0
        return self

In [113]:
from sklearn.linear_model import Ridge
class SurrogateRegressor(Ridge):
     def fit(self, X, y, sample_weight=None):
         X=X.flatten()
         return self

In [110]:
[class_names[i] for i in [3,7]]

['grain', 'trade']

In [114]:
#c = make_pipeline(pre, model)
#print(c.predict(corpus_val[idx]))
#print(model.predict(Xval[idx]))
#print(c.predict_proba(corpus_val))
#print(np.shape(Xval), Xval.ndim)
class_names=class_names[dataname]
class_names=[class_names[i] for i in [3,7]]
explainer = LimeTextExplainer(class_names=class_names, bow=False) # kernel_width=10)
print(type(sample_text[0]), type(sample_text))
RidgeSurrogate=SurrogateRegressor()
#model_regressor=LogisticRegression(C = 0.5, solver = "sag")
exp = explainer.explain_instance(sample_text, c.predict_proba, #num_features=np.prod(xval.shape), #
                                 num_features=len(re.split(r"\W+",sample_text)), \
                                 model_regressor=RidgeSurrogate) #LemnaRegressor())
exp.as_list() #explain_instance_with_data

<class 'str'> <class 'str'>

 corpus_txt[0], and its type and len:  asian exporters fear damage japan rift mounting trade friction and japan raised fears asia exporting nations that row inflict reaching economic damage businessmen and officials told reuter correspondents asian capitals move japan boost protectionist sentiment and lead curbs american imports products exporters that conflict hurt long run short term tokyo loss gain will impose mln dlrs tariffs imports japanese electronics goods april retaliation for japan alleged failure stick pact not sell semiconductors world markets below cost unofficial japanese estimates put impact tariffs billion dlrs and spokesmen for major electronics firms virtually halt exports products hit taxes wouldn business spokesman for leading japanese electronics firm matsushita electric industrial tariffs remain place for length time months will mean complete erosion exports goods subject tariffs tom murtha stock analyst tokyo office broker james capel

ValueError: Found input variables with inconsistent numbers of samples: [5000, 300]

In [91]:
class preprocessing:
    def __init__(self,subspace_dim, emb_size, emb_model):
        self.subspace_dim=subspace_dim
        self.emb_size=emb_size
        self.emb_model=emb_model
    def transform(self, corpus_txt):
        dim=subspace_dim
        emb_MODEL=self.emb_model        
        print('\n prepro corpus txt: ', type(corpus_txt))
        corpus_txt_str=' '.join(corpus_txt)
       # print('\n prepro corpus txt_str: ', type(corpus_txt_str))
       # corpus_subspace, words, words_emb, word_imp_no_rot, token = get_subspace_of_doc(corpus_txt, dim)
        corpus_subspaces = get_docs_subspaces(corpus_txt, emb_MODEL, self.emb_size, dim)
        return corpus_subspace

In [72]:
import scipy
import numpy as np
def monkeypath_itemfreq(sampler_indices):
   return zip(*np.unique(sampler_indices, return_counts=True))

scipy.stats.itemfreq=monkeypath_itemfreq
#import eli5
from eli5.lime import TextExplainer

te = TextExplainer(random_state=42)
te.fit(sample_text, c.predict_proba)
te.show_prediction(target_names=class_names[dataname])


 corpus_txt[0], and its type and len:  asian exporters fear damage  rift mounting trade friction and japan raised fears asia exporting nations that row  reaching economic damage businessmen and officials told reuter correspondents  capitals move japan boost  sentiment and lead curbs  imports products exporters that conflict  long run short term tokyo loss gain will impose  dlrs tariffs imports japanese electronics goods april  for japan  failure stick pact not sell semiconductors world markets  cost unofficial japanese estimates put impact tariffs billion  and spokesmen for major electronics firms virtually halt exports products  taxes wouldn business spokesman for leading japanese electronics firm matsushita  industrial  remain place for  time months will mean  erosion exports goods  tariffs tom murtha stock  tokyo office broker james capel and  businessmen and   aware seriousness threat japan serves warning senior  trade official  not named  had  trade surplus billion dlrs last year

ValueError: Found input variables with inconsistent numbers of samples: [5000, 5000, 300]

In [ ]:
#ptr = tokenizer(corpus_tr)
pte = tokenizer(corpus_val)
#ltr = {i:len(n) for i, n in enumerate(ptr)}
lte = {i:len(n) for i,n in enumerate(pte)}
#lte
#np.min(list(Counter(ltr).values())), 
print(np.min(list(Counter(lte).values())))
print([i for i, k in lte.items() if k==1])
print(df_te.shape)
#df_tr.iloc[3402], 
df_te.iloc[1875]
NLP = spacy.blank('en')
pred_val=model.prediction(Xval)
prova_val=model.predict_proba(Xval)
# {cat[i]: t for i, t in enumerate(ptr) if len(t)==1}

In [ ]:
# model['It']
#Model=model.prediction(corpus_val)
fname= 
xprotos, yprotos, rel = return_model(fname)
dim = xprotos.shape[-1]
D = xprotos.shape[-2]

In [ ]:
np.savez("./text.npz", a=a)